# Week 2 — Program Flow for LLM Calls

Covers: generators/`yield` (streaming), `async`/`await` basics, and building a simple CLI.

## 1. Generators — how streaming responses are consumed

In [ ]:
def mock_stream_response(full_text: str):
    """A generator that yields one word at a time — this is the *shape* of how a
    real streaming LLM API delivers tokens progressively instead of all at once."""
    for word in full_text.split():
        yield word + " "

print("Streaming output: ", end="")
for chunk in mock_stream_response("The extraction completed with high confidence."):
    print(chunk, end="", flush=True)
print()

## 2. `async`/`await` basics

In [ ]:
import asyncio

async def call_llm(name: str, delay: float) -> str:
    """Simulates an LLM call that takes `delay` seconds — e.g. network latency."""
    await asyncio.sleep(delay)
    return f"{name} finished after {delay}s"

async def main():
    # Awaiting one at a time — total time is the SUM of delays
    result = await call_llm("sequential-call", 0.2)
    print(result)

asyncio.run(main())

In [ ]:
async def main_concurrent():
    # asyncio.gather runs both concurrently — total time is the MAX of delays, not the sum
    results = await asyncio.gather(
        call_llm("call-A", 0.3),
        call_llm("call-B", 0.2),
    )
    for r in results:
        print(r)

asyncio.run(main_concurrent())

This is exactly why Week 4/5 lean on `async`/`await`: a multi-agent system calling three
tools or three LLMs sequentially is three times slower than necessary if those calls don't
depend on each other's results.

## 3. Building a simple CLI

In [ ]:
import argparse

def build_arg_parser():
    parser = argparse.ArgumentParser(description="SmartIntake-style CLI (demo)")
    parser.add_argument("--org", required=True, help="Fictional org, e.g. 'PharmaCore India'")
    parser.add_argument("--urgency", choices=["low", "medium", "high"], default="medium")
    return parser

parser = build_arg_parser()

# Simulating command-line input, since notebooks don't have a real terminal argv.
# In a real terminal you'd run:  python intake_cli.py --org "PharmaCore India" --urgency high
simulated_argv = ["--org", "PharmaCore India", "--urgency", "high"]
args = parser.parse_args(simulated_argv)

print(f"Received intake request from {args.org} at {args.urgency} urgency.")